## Imports

## Training

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, TextVectorization, Embedding, Dense, SimpleRNN, LSTM, GRU, RNN, SimpleRNNCell
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD AND PREPARE CSV DATA
# ==========================================
print("Loading data from CSV...")
csv_filename = "../data/IMDB Dataset.csv"

try:
    df = pd.read_csv(csv_filename)
except FileNotFoundError:
    print(f"Error: Could not find {csv_filename}. Please ensure it is in the same directory.")
    exit()

# Extract texts as a standard Python list
texts = df['review'].astype(str).tolist()

# Convert to float32 binary labels
labels = np.where(
    df['sentiment'].astype(str).str.lower().str.strip() == 'positive', 
    1.0, 
    0.0
).astype(np.float32)

# Split the data
print("Splitting data into train, validation, and test sets...")
train_texts, temp_texts, train_labels, temp_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)

# ==========================================
# 2. TEXT VECTORIZATION (PRE-PROCESSING)
# ==========================================
# Kept optimized for speed
max_tokens = 5000  
max_len = 80       

vectorize_layer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_len
)

print("Adapting TextVectorization layer to the training vocabulary...")
vectorize_layer.adapt(train_texts)

print("Pre-vectorizing the text data (this guarantees no graph errors and speeds up training)...")
# Transform the text strings into integer tensors BEFORE they go into the model
train_x = vectorize_layer(train_texts)
val_x = vectorize_layer(val_texts)
test_x = vectorize_layer(test_texts)

# Convert labels to tensors
train_y = tf.constant(train_labels)
val_y = tf.constant(val_labels)
test_y = tf.constant(test_labels)

# ==========================================
# 3. MODEL BUILDER FUNCTION
# ==========================================
def build_model(rnn_type="SimpleRNN"):
    model = Sequential()
    
    # Since data is already vectorized, the model takes integers (int32) of length max_len
    model.add(Input(shape=(max_len,), dtype="int32"))
    
    # We go straight into the Embedding layer
    model.add(Embedding(input_dim=max_tokens + 1, output_dim=32))
    
    if rnn_type == "SimpleRNN":
        model.add(RNN(SimpleRNNCell(16), return_sequences=False, return_state=False))
    elif rnn_type == "LSTM":
        model.add(LSTM(16))
    elif rnn_type == "GRU":
        model.add(GRU(16))
        
    model.add(Dense(16, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ==========================================
# 4. TRAINING, SAVING, AND CALLBACKS
# ==========================================
models_to_test = ["SimpleRNN", "LSTM", "GRU"]
epochs_to_run = 5 
batch_size = 256 

for architecture in models_to_test:
    print(f"\n==========================================")
    print(f"--- Training and Evaluating {architecture} ---")
    print(f"==========================================")
    
    model = build_model(rnn_type=architecture)
    
    early_stop = EarlyStopping(
        monitor='val_loss', 
        patience=2,          
        restore_best_weights=True
    )
    
    model_checkpoint = ModelCheckpoint(
        filepath=f"best_model_{architecture}.keras",
        monitor='val_loss',
        save_best_only=True
    )
    
    # Train using the pre-vectorized tensors (train_x, train_y)
    history = model.fit(
        x=train_x, 
        y=train_y,
        validation_data=(val_x, val_y),
        epochs=epochs_to_run,
        batch_size=batch_size,
        callbacks=[early_stop, model_checkpoint]
    )
    
    print(f"\nEvaluating {architecture} on Test Data...")
    test_loss, test_acc = model.evaluate(test_x, test_y, batch_size=batch_size)
    print(f"Test Accuracy for {architecture}: {test_acc:.4f}")

    model.save(f"final_{architecture}_model.keras")

    print(f"\nTesting {architecture} with custom reviews:")
    custom_reviews = [
        "This movie was an absolute triumph. The direction was brilliant and the acting was top notch.",
        "I regret spending money on this. The plot made no sense and it was incredibly boring."
    ]
    # We manually pass the custom reviews through the vectorizer before predicting
    custom_reviews_vectorized = vectorize_layer(custom_reviews)
    
    predictions = model.predict(custom_reviews_vectorized)
    for review, pred in zip(custom_reviews, predictions):
        sentiment = "Positive" if pred[0] > 0.5 else "Negative"
        print(f"Review: '{review[:50]}...' -> Prediction: {sentiment} ({pred[0]:.4f})")

print("\n--- Lab Execution Complete ---")

Loading data from CSV...
Splitting data into train, validation, and test sets...
Adapting TextVectorization layer to the training vocabulary...
Pre-vectorizing the text data (this guarantees no graph errors and speeds up training)...

--- Training and Evaluating SimpleRNN ---
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 519s 3s/step - accuracy: 0.6497 - loss: 0.6151 - val_accuracy: 0.7388 - val_loss: 0.5535
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 488s 3s/step - accuracy: 0.8038 - loss: 0.4413 - val_accuracy: 0.8108 - val_loss: 0.4338
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 519s 3s/step - accuracy: 0.8437 - loss: 0.3694 - val_accuracy: 0.7992 - val_loss: 0.4333
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 546s 3s/step - accuracy: 0.8643 - loss: 0.3302 - val_accuracy: 0.8036 - val_loss: 0.4494
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 540s 3s/step - accuracy: 0.8829 - loss: 0.2940 - val_accuracy: 0.8088 - val_loss: 0.4731

Evaluating SimpleRNN on Test Data...
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 268ms/step - ac